In [1]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE


def create_tf_dataset(
    image_paths,
    mask_paths,
    batch_size=16,
    shuffle=False
):

    def load_pair(image_path, mask_path):

        # -------------------------
        # Load image
        # -------------------------
        image = tf.io.read_file(image_path)

        image = tf.image.decode_png(
            image,
            channels=1
        )

        # -------------------------
        # Load mask
        # -------------------------
        mask = tf.io.read_file(mask_path)

        mask = tf.image.decode_png(
            mask,
            channels=1
        )

        # -------------------------
        # Resize
        # -------------------------

        image = tf.image.resize(
            image,
            [128, 128],
            method=tf.image.ResizeMethod.BILINEAR
        )

        mask = tf.image.resize(
            mask,
            [128, 128],
            method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
        )

        # -------------------------
        # Normalize image
        # -------------------------

        image = tf.cast(
            image,
            tf.float32
        ) / 255.0

        # -------------------------
        # Binary mask
        # -------------------------

        mask = tf.cast(
            mask > 127,
            tf.float32
        )

        return image, mask

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            [str(p) for p in image_paths],
            [str(p) for p in mask_paths]
        )
    )

    if shuffle:

        dataset = dataset.shuffle(
            buffer_size=len(image_paths),
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_pair,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(
        batch_size
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset

2026-10-06 08:14:28.112261: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-06 08:14:28.265329: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791274468.378524    1981 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791274468.412962    1981 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-06 08:14:28.810296: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr